# 07 · Transformer (PatchTST) frente a LSTM: Recall y latencia
Pedido 5.2 del especialista. Mismas entradas, misma pérdida (WFL), misma partición y mismo evaluador. Se mide además la **latencia de inferencia de una ventana** en CPU (despliegue Edge) y GPU.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos, correr_variante
from s4.registry import leer, resumir_semillas
from s4.train import configurar_gpu, medir_latencia
from s4 import features as ft
configurar_gpu()
cfg = cargar_config('../config/ablacion_2025_2026.yaml')
datos = preparar_datos(cfg)
CONJUNTO = "E_completo"      # <-- el mejor de la ablación

In [ ]:
PATCH = {"tipo": "patchtst", "patch_len": 16, "stride": 8, "d_model": 64, "n_heads": 4, "n_capas": 2, "d_ff": 128, "dropout": 0.2}
salida_lstm = correr_variante(cfg, datos, ft.CONJUNTOS[CONJUNTO], "LSTM apilado", "transformer", guardar_modelo=True)
salida_ptst = correr_variante(variante(cfg, modelo=PATCH), datos, ft.CONJUNTOS[CONJUNTO], "PatchTST", "transformer", guardar_modelo=True)

In [ ]:
lat = []
for nombre, (filas, modelos, parts) in [("LSTM apilado", salida_lstm), ("PatchTST", salida_ptst)]:
    m = modelos[0][1]
    for disp in ["/CPU:0", "/GPU:0"]:
        lat.append({"modelo": nombre, **medir_latencia(m, parts["val"], dispositivo=disp)})
lat = pd.DataFrame(lat); lat

In [ ]:
reg = leer(ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv", grupo="transformer", particion="val")
tabla = resumir_semillas(reg, por=("modelo",), metricas=("RMSE_global","RMSE_evento","POD","Precision","HSS","POD_inicio"))
tabla = tabla.merge(lat[lat["dispositivo"]=="/CPU:0"][["modelo","lat_ms_mediana","parametros"]], on="modelo")
tabla.to_csv(ruta(cfg,"reportes")/"tables/tabla_patchtst_vs_lstm.csv", index=False); tabla.round(4)